# Stage 0 analysis — free-choice spreading of alternatives

Reads the artifacts written by `python -m src.experiments.run --config <run.yaml>`.
It does **not** run models: every number here comes from a cached, provenance-stamped
artifact, and the pooling guard refuses to combine artifacts across devices or dtypes.

**Primary test** is `(chose − yoked) × |diff|`, predicted negative. A main effect of
agency is reported but is *not* the test: on its own it is fully consistent with
context-window sensitivity, which is how the prior version of this claim was
eliminated. See `preregistration.md` §2.

The contrast tables below are the table view that accompanies every figure, so
series identity is never conveyed by colour alone.

In [ ]:
import sys, json, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import arviz as az
import numpy as np
import pandas as pd

from src.analysis import mixed, plots
from src.analysis.reliability import evaluate_gates, item_scores
from src.config import load_config
from src.experiments import pass_a as stage_a, pass_b as stage_b, pass_c as stage_c
from src.provenance import assert_poolable, read_parquet, provenance_of

plots.use_paper_defaults()
pd.set_option("display.width", 160, "display.max_columns", 40)

# Point this at the run you want to inspect.
CONFIG = ROOT / "configs" / "stage0_qwen2.5-3b.yaml"
cfg = load_config(CONFIG)
cfg = cfg.model_copy(update={"artifacts_dir": ROOT / "artifacts"})
print(cfg.model.name, "| config hash", cfg.hash())

## 1. Provenance

All reported numbers must originate on the run machine (CUDA, bf16, pinned revision,
clean tree). `assert_reportable` fails otherwise; smoke artifacts from the M1 are
rejected here by design.

In [ ]:
pass_a_frame = read_parquet(stage_a.artifact_path(cfg))
pairs = read_parquet(stage_b.artifact_path(cfg))
pass_c_frame = read_parquet(stage_c.artifact_path(cfg))

# Hard guard: raises if these came off different devices or dtypes.
assert_poolable([pass_a_frame, pairs, pass_c_frame], context="notebook load")

prov = provenance_of(pass_c_frame)
pd.Series(prov).to_frame("value")

In [ ]:
from src.provenance import Provenance, ProvenanceError, assert_reportable

try:
    assert_reportable(Provenance(**prov))
    print("REPORTABLE: run-machine artifact, safe to put in the paper.")
except ProvenanceError as exc:
    print("NOT REPORTABLE (expected for M1 smoke runs):")
    print(exc)

## 2. Pass A — validity, reliability, SESOI

Two distinct properties, and conflating them is the mistake to avoid:

- **Validity** (`ρ` between ascending and reversed-descending, per template) — does the
  model read the scale at all? `ρ ≥ 0.6` is the **sole categorical exclusion**.
  A strongly *negative* `ρ` means the model ignores the anchor definition and answers on
  a fixed higher-is-better mapping.
- **Reliability** (`ICC(C,1)` across the 5 templates) — how noisy is it? Never a
  scientific halt; a tripwire below 0.4, otherwise an input to the power simulation.

`SESOI = 0.15 × σ_between`, fixed before Pass C ran.

In [ ]:
gate = evaluate_gates(cfg, pass_a_frame)
print(gate.summary())
sesoi = gate.sesoi_primary
gate.validity.round(4)

In [ ]:
print("digit_mass — probability landing on the 1–9 digit tokens.")
print("Low values mean the readout position is not the digit position.")
display(pass_a_frame.groupby("template")["digit_mass"].describe().round(4))

print("\nExpected value vs argmax: EV must not collapse to the modal digit,")
print("or the graded information the DV depends on has been discarded.")
print((pass_a_frame["rating"] - pass_a_frame["rating_argmax"]).abs().describe().round(4))

## 3. Pass B — pair construction

Difficulty is *selected* on `mean(T1–T3)` and *analysed* on `mean(T4–T5)`: selecting on a
noisy `|diff|` and regressing on the same noisy `|diff|` would leave the regressor
regression-contaminated.

Difficult and easy pools are **matched on mean pair rating** by design, not by covariate —
otherwise ceiling compression alone could manufacture the difficulty interaction.

In [ ]:
diag = stage_b.pair_diagnostics(pairs)
display(diag.round(4))
print("matching quality (|difficult − easy| mean pair rating, within matched set):")
print({k: round(v, 4) for k, v in diag.attrs.items()})

print("\nitem reuse (cap = %d), and items in both levels (must be 0):" % cfg.pass_b.max_uses_per_item)
long = pd.concat([
    pairs[["item1_id", "difficulty"]].rename(columns={"item1_id": "item"}),
    pairs[["item2_id", "difficulty"]].rename(columns={"item2_id": "item"}),
])
print("max uses:", long["item"].value_counts().max(),
      "| items in both levels:", int((long.groupby("item")["difficulty"].nunique() > 1).sum()))

## 4. Pass C — descriptives and position bias

`spread = (designated_post − designated_pre) − (other_post − other_pre)`, bounded ±16 on a
1–9 scale. Realistic magnitudes are around ±2; read it as points of divergence between the
two options, not as a rating.

Choice is elicited per (pair, template, option order) and yoking is defined **within**
order, so a flip across option orders does not desynchronise `chose` from `yoked`. The flip
rate is the natural measure of how much of the choice is position rather than preference.

In [ ]:
display(pass_c_frame.pivot_table(index="condition", columns="difficulty",
                                 values="spread", aggfunc=["mean", "std", "count"]).round(3))

choice_diag = stage_c.choice_diagnostics(pass_c_frame)
display(choice_diag.round(4))
print("overall flip rate across option order:",
      round(choice_diag.attrs.get("overall_flip_rate", float("nan")), 4))
print("\nrate at which the model's pick was the higher-pre-rated item:")
chose = pass_c_frame[pass_c_frame.condition == "chose"]
print(round(float((chose["designated_pre"] > chose["other_pre"]).mean()), 4))

## 5. Mixed-effects model

Cell-means parameterization, so every planned contrast is a plain difference of posterior
draws:

```
mu = b_cond[c] + b_slope[c]·diff_z + b_order·order + u_pair[p] + u_template[t]
```

`diff_z` is z-scored within model, so the interaction coefficient is in **spread points per
SD of |diff|** and is directly comparable to the SESOI.

In [ ]:
design = mixed.prepare_design(cfg, pass_c_frame)
print(f"|diff| centring: mean={design.diff_mean:.4f} sd={design.diff_sd:.4f}")

out_dir = cfg.artifact_dir("analysis")
posterior_path = out_dir / f"posterior_{cfg.hash()}.nc"
if posterior_path.exists():
    idata = az.from_netcdf(str(posterior_path))
    print("loaded cached posterior:", posterior_path.name)
else:
    print("no cached posterior; fitting")
    idata = mixed.fit(cfg, design, with_item=False, progressbar=True)

conv = mixed.convergence(cfg, idata)
bad = conv[~(conv["rhat_ok"] & conv["ess_ok"])]
print(f"convergence failures: {len(bad)}")
display(bad if len(bad) else conv.head(12).round(4))

## 6. Planned contrasts

| contrast | what it isolates |
|---|---|
| `chose − yoked` × \|diff\| | **PRIMARY** — agency, designation held constant |
| `chose − 3p-yoked` | authorship + self-relevance, information constant |
| `3p-yoked − yoked` | information effect, designation constant |
| `yoked − random` | selection artifact |
| `3p-random − random` | pure context effect — the rebuttal's mechanism |

The rebuttal's account predicts the effect lives in `3p-yoked − yoked` and
`3p-random − random` and that the **primary interaction is null**. Consistency restoration
predicts the interaction is present regardless of those two.

In [ ]:
contrasts = mixed.contrast_table(cfg, idata, sesoi)
print(f"SESOI = {sesoi:.4f} spread points per SD of |diff|  (0.15 × σ_between)")
print(f"secondary fixed anchor = {cfg.analysis.sesoi_raw_secondary} raw rating points\n")

print("INTERACTION with |diff| (the primary term):")
display(contrasts[contrasts.term == "slope"].round(4).set_index("name"))
print("MAIN EFFECT at mean |diff| (reported, never primary):")
display(contrasts[contrasts.term == "intercept"].round(4).set_index("name"))

In [ ]:
primary = contrasts[(contrasts.name == mixed.PRIMARY) & (contrasts.term == "slope")].iloc[0]
print(f"PRIMARY TEST  {mixed.PRIMARY} × |diff|  ->  {primary['decision'].upper()}")
print(f"  median {primary['median']:+.4f}   "
      f"{int(cfg.analysis.hdi_prob*100)}% HDI [{primary['hdi_low']:+.4f}, {primary['hdi_high']:+.4f}]")
print(f"  P(<0) = {primary['p_negative']:.3f}   exceeds SESOI: {bool(primary['exceeds_sesoi'])}   "
      f"inside ROPE: {bool(primary['inside_rope'])}")

agree = mixed.artifact_agreement(contrasts)
if agree:
    print("\nArtifact cross-check — two routes to the same quantity; they should agree.")
    print(f"  {agree['route_a']} = {agree['estimate_a']:+.4f}")
    print(f"  {agree['route_b']} = {agree['estimate_b']:+.4f}")
    print(f"  discrepancy = {agree['discrepancy']:+.4f}")
    print("  A large discrepancy means Stage 0 is uninterpretable, not a pass or a fail.")

## 7. Figures

Interaction plot: fitted posterior lines with HDI bands, plus binned means with 95%
intervals on the primary panel. Every series carries a direct label, so identity does not
depend on colour.

In [ ]:
fig = plots.interaction_plot(cfg, design, idata)
plots.save(fig, out_dir / "figures" / f"nb_interaction_{cfg.hash()}.png")
fig

In [ ]:
fig = plots.forest_plot(cfg, contrasts, sesoi, term="slope")
plots.save(fig, out_dir / "figures" / f"nb_forest_slope_{cfg.hash()}.png")
fig

In [ ]:
fig = plots.forest_plot(cfg, contrasts, sesoi, term="intercept")
plots.save(fig, out_dir / "figures" / f"nb_forest_intercept_{cfg.hash()}.png")
fig

## 8. Robustness — item random effect via multi-membership

The DV is per-pair while each pair holds two items, so the item effect enters as
`u_item[item1] + u_item[item2]`. Item reuse is capped at 2 pairs, so this should agree
closely with the primary model; the discrepancy is reported either way.

In [ ]:
item_path = out_dir / f"posterior_item_{cfg.hash()}.nc"
if item_path.exists():
    idata_item = az.from_netcdf(str(item_path))
else:
    print("fitting the robustness model (rerun with --robustness to cache it)")
    idata_item = mixed.fit(cfg, design, with_item=True, progressbar=True)

c_item = mixed.contrast_table(cfg, idata_item, sesoi)
comp = (
    contrasts[contrasts.term == "slope"].set_index("name")[["median", "hdi_low", "hdi_high"]]
    .join(c_item[c_item.term == "slope"].set_index("name")[["median", "hdi_low", "hdi_high"]],
          lsuffix="_primary", rsuffix="_item_re")
)
comp["delta_median"] = comp["median_item_re"] - comp["median_primary"]
comp.round(4)

## 9. The ladder

Whether the interaction emerges with scale is a primary descriptive result regardless of
the gate outcome. Stage 1 is entered only if **at least two non-excluded models pass**.

Models excluded for polarity validity are reported as *not having demonstrated that they
read the scale* — that is not evidence about the hypotheses in either direction.

In [ ]:
LADDER = [
    "stage0_qwen2.5-0.5b.yaml",
    "stage0_qwen2.5-1.5b.yaml",
    "stage0_qwen2.5-3b.yaml",
    "stage0_gemma-2-2b.yaml",
    "stage0_llama-3.2-3b.yaml",
]

rows = []
for name in LADDER:
    c = load_config(ROOT / "configs" / name).model_copy(
        update={"artifacts_dir": ROOT / "artifacts"})
    res = c.artifact_dir("analysis") / f"results_{c.hash()}.json"
    if not res.exists():
        rows.append({"model": c.model.name, "status": "not run"})
        continue
    r = json.loads(res.read_text())
    g, p = r.get("gates", {}), r.get("primary", {})
    rows.append({
        "model": c.model.name,
        "status": r.get("outcome", "?"),
        "device": r.get("provenance", {}).get("device"),
        "median_rho": g.get("median_rho"),
        "icc_c1": (g.get("icc_all") or {}).get("icc_c1"),
        "sigma_between": g.get("sigma_between"),
        "sesoi": g.get("sesoi_primary"),
        "power_at_sesoi": (r.get("power") or {}).get("power_at_sesoi"),
        "interaction": p.get("median"),
        "hdi_low": p.get("hdi_low"),
        "hdi_high": p.get("hdi_high"),
    })

ladder = pd.DataFrame(rows)
n_pass = int((ladder["status"] == "primary-pass").sum())
print(f"models passing the primary test: {n_pass}")
print("GATE: proceed to Stage 1" if n_pass >= 2 else
      "GATE: do NOT proceed — Stage 0 is a kill gate and the later-stage code is not written.")
ladder.round(4)